<div class="blog-language-switch" role="group" aria-label="文章语言">
<a href="../../Machine-Learning/17-online-bandits-reinforcement-learning.html" lang="en" hreflang="en">English</a>
<span aria-current="page">中文</span>
</div>

[返回机器学习总览](Machine%20Learning.html)


## **在线学习、Bandit 与强化学习**

前面的多数章节都假设：模型拟合之前，已经存在一个固定的训练集。**序列学习器（sequential learner）**则必须在数据仍不断到达时进行预测或采取行动。它的决策可能决定哪些反馈能够被观察、改变后续状态，甚至让人和系统承担真实后果。因此，核心问题不再只是“哪个模型最适合这份数据”，而是：

> 做出决策前能够获得什么信息，决策后会揭示什么反馈，当前行动是否会改变未来机会？

这三个问题区分了几种彼此相关却并不等价的建模形式：

| 建模形式 | 行动前的信息 | 行动后的反馈 | 行动是否影响下一上下文或状态？ | 主要目标 |
|---|---|---|---|---|
| 批量监督学习 | 固定的带标签数据集 | 不要求在线反馈 | 否 | 泛化风险 |
| 在线学习 | 当前样本或损失上下文 | 完整目标或完整损失信息 | 通常视为外生变量 | 相对比较器的遗憾 |
| 多臂 Bandit | 无上下文，或只有时间与历史 | 只观察所选行动的奖励 | 不建模长期状态影响 | Bandit 遗憾 |
| 上下文 Bandit | 当前上下文 $x_t$ | 只观察所选行动的奖励 | 上下文被视为外生变量 | 上下文策略价值或遗憾 |
| 强化学习 | 当前状态或观测 | 奖励和下一状态 | 是 | 期望长期回报 |

<div class="diagram-scroll">

![从批量预测到强化学习的建模谱系。](assets/sequential-feedback-spectrum.svg){fig-alt="批量学习、在线预测、Bandit、上下文 Bandit 和马尔可夫决策过程依次具有更不完整的反馈，以及行动对未来状态更强的影响。"}

</div>

采用更强大的建模形式并不一定更好。强化学习会额外引入延迟信用分配、探索风险、非独立数据和困难的评估问题。如果行动不会影响未来机会，上下文 Bandit 往往更简单、统计效率也更高；如果每个行动的损失都会被揭示，完整信息在线学习就应利用这些额外反馈。

### **在线学习协议**

在线学习研究一系列轮次 $t=1,\ldots,T$。一种常见的监督学习协议是：

1. 学习器观察 $x_t$，并使用 $w_t$ 选择参数或给出预测；
2. 系统揭示结果 $y_t$ 或损失函数 $\ell_t$；
3. 学习器承担损失 $\ell_t(w_t)$，再更新为 $w_{t+1}$。

当前预测必须在看到当前结果**之前**完成。每获得一条记录就更新一次批量模型，在计算上属于增量训练；但如果模型在预测该记录前已经看到了它自己的目标，这就不是有效的在线评估。

<div class="diagram-scroll">

![在线协议中的观察、决策、反馈揭示与更新。](assets/online-learning-protocol.svg){fig-alt="每轮在线学习先观察上下文，在结果揭示前给出预测，承担损失后才更新参数供下一轮使用。"}

</div>

在线学习不要求观测独立同分布。数据序列可以是随机的、对抗性的、季节性的或不断漂移的。这种灵活性使性能标准从估计单一总体风险，转变为把累积决策与一个定义明确的参照进行比较。

#### **序列预测与遗憾**

对于比较器类别 $\mathcal W$，**静态遗憾（static regret）**定义为

$$
R_T
=
\sum_{t=1}^{T}\ell_t(w_t)
-
\min_{w\in\mathcal W}
\sum_{t=1}^{T}\ell_t(w).
$$

比较器是事后选出的一个固定决策。当 $R_T/T\to 0$ 时，学习器被称为**无遗憾（no-regret）**：尽管它必须按顺序行动，但相对于事后最佳固定比较器，其平均额外损失会趋近于零。

这并不意味着学习器能够跟踪不断变化的最优解。**动态遗憾（dynamic regret）**

$$
R_T^{\mathrm{dyn}}
=
\sum_{t=1}^{T}\ell_t(w_t)
-
\sum_{t=1}^{T}\ell_t(w_t^\star)
$$

采用随时间变化的比较器。只有限制比较器的变化幅度后，它才有意义，例如限制路径长度 $\sum_t\lVert w_t^\star-w_{t-1}^\star\rVert$。如果没有这种约束，比较器可以在每次观测后任意变化，任何学习器都无法与之竞争。

<div class="diagram-scroll">

![静态遗憾与动态遗憾使用不同的比较器。](assets/online-regret-comparators.svg){fig-alt="静态遗憾把在线学习器与一个事后固定决策比较；动态遗憾则与变化受到约束的时变决策序列比较。"}

</div>

遗憾是运行过程中的相对比较，而不是测试集准确率。它取决于时间跨度、损失函数、比较器类别、反馈协议和对数据序列的假设；报告结果时应同时说明这五项。


#### **在线梯度下降与在线 Perceptron**

对于可行集合 $\mathcal W$ 上的凸损失，**投影在线梯度下降（projected OGD）**执行

$$
w_{t+1}
=
\Pi_{\mathcal W}
\left(
w_t-\eta_t\nabla\ell_t(w_t)
\right),
$$

其中 $\Pi_{\mathcal W}$ 把不受约束的更新投影回可行集合。投影并非装饰性步骤：遗憾界需要控制决策集合的直径，而实际约束也可能要求权重、预算或概率处于有效范围内。

如果 $\mathcal W$ 的直径为 $D$，所有梯度范数不超过 $G$，并取 $\eta=D/(G\sqrt T)$，标准凸分析可得

$$
R_T\le DG\sqrt T,
\qquad
\frac{R_T}{T}\le\frac{DG}{\sqrt T}.
$$

证明过程会展开 $\lVert w_{t+1}-w\rVert^2$，利用投影的非扩张性与凸性，再让各轮距离项望远镜式相消。这个更新看起来与随机梯度下降相似，但解释不同：SGD 在抽样假设下估计批量期望目标，而 OGD 即使面对非随机损失序列，也是在控制序列遗憾。

<details>
<summary><strong>Python：运行投影 OGD 并计算实际静态遗憾</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(17)
horizon, dimensions = 2_000, 5

# Bound every context norm so the loss gradients cannot grow without control.
contexts = rng.normal(size=(horizon, dimensions))
contexts /= np.maximum(1.0, np.linalg.norm(contexts, axis=1, keepdims=True))
true_weight = np.array([1.5, -1.0, 0.7, 0.2, -0.4])
targets = contexts @ true_weight + rng.normal(scale=0.12, size=horizon)

radius = 3.0
weight = np.zeros(dimensions)
online_losses = []

for time_index, (context, target) in enumerate(zip(contexts, targets), start=1):
    prediction = context @ weight
    error = prediction - target
    online_losses.append(0.5 * error**2)

    # The current target is used only after the current prediction is scored.
    gradient = error * context
    learning_rate = 0.8 / np.sqrt(time_index)
    candidate = weight - learning_rate * gradient

    # Euclidean projection onto ||w||_2 <= radius.
    candidate_norm = np.linalg.norm(candidate)
    weight = candidate * min(1.0, radius / max(candidate_norm, 1e-12))

# The static comparator may inspect the full sequence, but must use one weight.
comparator, *_ = np.linalg.lstsq(contexts, targets, rcond=None)
comparator *= min(1.0, radius / max(np.linalg.norm(comparator), 1e-12))
comparator_losses = 0.5 * (contexts @ comparator - targets) ** 2

static_regret = np.sum(online_losses) - np.sum(comparator_losses)
print("online cumulative loss:", round(float(np.sum(online_losses)), 3))
print("best fixed cumulative loss:", round(float(np.sum(comparator_losses)), 3))
print("static regret:", round(float(static_regret), 3))
print("average regret per round:", round(float(static_regret / horizon), 5))
print("distance to comparator:", round(float(np.linalg.norm(weight - comparator)), 3))
```

</details>

对于二元分类，**在线 Perceptron**预测 $\hat y_t=\operatorname{sign}(w_t^\top x_t)$，并且只在出错后更新：

$$
w_{t+1}
=
w_t+y_tx_t
\quad\text{if }y_tw_t^\top x_t\le 0.
$$

假设存在单位向量 $u$，使间隔满足 $y_tu^\top x_t\ge\gamma>0$，且 $\lVert x_t\rVert\le R$。经典的错误次数上界为

$$
M\le\left(\frac{R}{\gamma}\right)^2.
$$

每次错误都会使对齐程度 $u^\top w$ 至少增加 $\gamma$，而参数平方范数至多增加 $R^2$，结合二者即可限制错误总数。该上界是确定性的，也不依赖序列顺序；但数据不可分时它便不再成立，此时更适合使用间隔损失、正则化、参数平均或概率在线模型。

<details>
<summary><strong>Python：在具有正间隔的数据流上验证 Perceptron 更新</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(23)
separator = np.array([0.8, -0.5, 0.3])
separator /= np.linalg.norm(separator)

examples, labels = [], []
while len(examples) < 1_000:
    point = rng.normal(size=3)
    point /= max(1.0, np.linalg.norm(point))
    signed_distance = point @ separator

    # Reject points close to the boundary to construct a positive margin.
    if abs(signed_distance) >= 0.25:
        examples.append(point)
        labels.append(1 if signed_distance > 0 else -1)

examples = np.asarray(examples)
labels = np.asarray(labels)
order = rng.permutation(len(labels))

weight = np.zeros(3)
mistakes = 0
for index in order:
    if labels[index] * (weight @ examples[index]) <= 0:
        weight += labels[index] * examples[index]
        mistakes += 1

margin = np.min(labels * (examples @ separator))
max_radius = np.max(np.linalg.norm(examples, axis=1))
mistake_bound = (max_radius / margin) ** 2

predictions = np.where(examples @ weight >= 0, 1, -1)
print("observed mistakes:", mistakes)
print("theoretical upper bound:", round(float(mistake_bound), 1))
print("stream training accuracy:", round(float(np.mean(predictions == labels)), 3))
print("cosine with true separator:", round(float(weight @ separator / np.linalg.norm(weight)), 3))
```

</details>

在线更新反应迅速，却也可能追逐噪声。学习率计划、遗忘因子、滑动窗口、变化点告警、自适应遗憾和显式重置策略，应根据环境变化速度选择。评估必须采用**预测式顺序评估（prequential evaluation）**：先预测并记录损失，然后才能利用该观测学习。


### **多臂 Bandit**

随机 $K$ 臂 Bandit 包含未知的奖励分布 $\nu_1,\ldots,\nu_K$，其均值分别为 $\mu_1,\ldots,\mu_K$。第 $t$ 轮中，学习器选择臂 $A_t$，并且只观察该臂的奖励 $R_t(A_t)$；未选择臂的奖励属于不可见的反事实结果。

若 $\mu^\star=\max_a\mu_a$，期望伪遗憾为

$$
\bar R_T
=
T\mu^\star
-
\mathbb E\left[\sum_{t=1}^{T}\mu_{A_t}\right]
=
\sum_{a=1}^{K}\Delta_a\,\mathbb E[N_a(T)],
$$

其中 $\Delta_a=\mu^\star-\mu_a$，$N_a(T)$ 表示臂 $a$ 被选择的次数。这个分解说明遗憾来自选择次优臂，但为了识别最优臂，又必须进行一定数量的次优尝试。

<div class="diagram-scroll">

![完整信息反馈与 Bandit 反馈会揭示不同的结果。](assets/bandit-feedback-exploration.svg){fig-alt="完整信息学习能够观察所有行动的损失，而 Bandit 只观察所选行动的奖励，因此必须主动探索隐藏的备选项。"}

</div>

当一次决策不会改变模型中的未来状态时，Bandit 可用于选择单个推荐、通知、广告或治疗方案。延迟奖励、用户间干扰、库存约束和重复曝光都可能破坏这种单步简化。

#### **探索与利用**

**利用（exploitation）**选择当前估计最好的行动；**探索（exploration）**则选择不确定或表面较差的行动，因为结果可能改善未来决策。纯贪心策略可能因早期幸运奖励而永久锁定错误臂；均匀探索则可能在不确定性已经消除后继续浪费试验。

合适的探索方法取决于不确定性模型：

- 随机平稳 Bandit 可以使用集中不等式或贝叶斯后验；
- 对抗性 Bandit 需要 EXP3 等算法，而不能直接使用随机 UCB；
- 奖励漂移需要折扣、窗口、重启或变化检测；
- 即使无约束探索能降低统计遗憾，安全约束也可能禁止它。

时间跨度同样重要。只有在剩余决策足够多、能够弥补当前机会成本时，探索信息才具有价值。

#### **Epsilon-Greedy、UCB 与 Thompson Sampling**

**Epsilon-greedy** 以 $1-\epsilon$ 的概率选择经验均值最好的臂，以 $\epsilon$ 的概率进行均匀探索。它直观透明，但探索方向不取决于不确定性。固定 $\epsilon$ 在渐近意义下会产生线性遗憾；递减计划能够改善遗憾，却可能在分布漂移后适应过慢。

对于有界随机奖励，**UCB1** 选择

$$
A_t
=
\arg\max_a
\left[
\hat\mu_a(t)
+
\sqrt{\frac{2\log t}{N_a(t)}}
\right].
$$

第一项利用估计奖励；置信奖励项会对很少采样的臂给出更高分数。这体现了**面对不确定性的乐观原则（optimism under uncertainty）**：只要某个臂仍有可能很好，就暂时按照其置信上界行动。

对于 Bernoulli 奖励，**Thompson sampling** 维护

$$
\mu_a\mid\mathcal H_t
\sim
\operatorname{Beta}(\alpha_a,\beta_a),
$$

从每个后验中抽取 $\tilde\mu_a$，再选择 $\arg\max_a\tilde\mu_a$。一个可信的臂被选中的概率，大致与其成为最优臂的后验概率成比例。

<div class="diagram-scroll">

![Epsilon-greedy、UCB 和 Thompson sampling 对不确定性的处理方式不同。](assets/bandit-algorithm-beliefs.svg){fig-alt="Epsilon-greedy 使用随机探索，UCB 加入置信奖励，Thompson sampling 则选择后验抽样值最大的臂。"}

</div>

<details>
<summary><strong>Python：比较 epsilon-greedy、UCB 与 Thompson sampling</strong></summary>

```python
import numpy as np

arm_means = np.array([0.25, 0.35, 0.50, 0.55])
horizon = 2_000
repetitions = 60


def run_bandit(algorithm, seed):
    rng = np.random.default_rng(seed)
    counts = np.zeros(len(arm_means), dtype=int)
    successes = np.zeros(len(arm_means))
    chosen_arms = np.empty(horizon, dtype=int)

    for time_index in range(horizon):
        if algorithm == "epsilon-greedy":
            empirical_means = successes / np.maximum(counts, 1)
            if rng.random() < 0.10 or np.any(counts == 0):
                available = np.flatnonzero(counts == 0)
                arm = int(rng.choice(available if len(available) else len(arm_means)))
            else:
                arm = int(np.argmax(empirical_means))

        elif algorithm == "ucb":
            if np.any(counts == 0):
                arm = int(np.flatnonzero(counts == 0)[0])
            else:
                empirical_means = successes / counts
                bonus = np.sqrt(2.0 * np.log(time_index + 1) / counts)
                arm = int(np.argmax(empirical_means + bonus))

        elif algorithm == "thompson":
            samples = rng.beta(1.0 + successes, 1.0 + counts - successes)
            arm = int(np.argmax(samples))
        else:
            raise ValueError("unknown algorithm")

        reward = rng.binomial(1, arm_means[arm])
        counts[arm] += 1
        successes[arm] += reward
        chosen_arms[time_index] = arm

    pseudo_regret = np.sum(np.max(arm_means) - arm_means[chosen_arms])
    late_optimal_rate = np.mean(chosen_arms[-300:] == np.argmax(arm_means))
    return pseudo_regret, late_optimal_rate


for algorithm in ["epsilon-greedy", "ucb", "thompson"]:
    results = np.array(
        [run_bandit(algorithm, seed=10_000 + run) for run in range(repetitions)]
    )
    print(
        f"{algorithm:15s}",
        "mean regret =", round(float(results[:, 0].mean()), 1),
        "late optimal-arm rate =", round(float(results[:, 1].mean()), 3),
    )
```

</details>

一次模拟不能构成算法排名。应改变均值差距、时间跨度、奖励分布族、先验、漂移和调参值，并报告多个独立运行的遗憾分布，而不是只展示最好的随机种子。

#### **上下文 Bandit**

上下文 Bandit 在行动前观察 $x_t$，并学习策略 $\pi(a\mid x)$。在线性奖励模型下，

$$
\mathbb E[R_t(a)\mid x_t]
=
x_t^\top\theta_a,
$$

**LinUCB** 为每个行动维护岭回归估计，并选择

$$
A_t
=
\arg\max_a
\left[
x_t^\top\hat\theta_a
+
\alpha\sqrt{x_t^\top A_a^{-1}x_t}
\right].
$$

这里的不确定性奖励取决于上下文：同一个行动对于某类用户可能已被充分了解，对于另一类用户却仍然高度不确定。

<div class="diagram-scroll">

![上下文 Bandit 在选择一个行动前先观察上下文。](assets/contextual-bandit-protocol.svg){fig-alt="上下文进入策略，系统选择一个行动，只观察该行动的奖励，再更新所选行动的模型。"}

</div>

<details>
<summary><strong>Python：实现 LinUCB 并与无上下文 UCB 比较</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(31)
horizon, dimensions, actions = 2_500, 4, 3
true_parameters = np.array(
    [
        [0.9, -0.6, 0.2, 0.0],
        [-0.5, 0.8, 0.0, 0.4],
        [0.1, -0.2, 0.9, -0.7],
    ]
)

contexts = rng.normal(size=(horizon, dimensions))
expected_rewards = contexts @ true_parameters.T


def run_linucb(alpha=0.8):
    covariance = np.repeat(np.eye(dimensions)[None, :, :], actions, axis=0)
    reward_sum = np.zeros((actions, dimensions))
    selected = []

    for context, means in zip(contexts, expected_rewards):
        scores = []
        for action in range(actions):
            inverse = np.linalg.inv(covariance[action])
            estimate = inverse @ reward_sum[action]
            bonus = alpha * np.sqrt(context @ inverse @ context)
            scores.append(context @ estimate + bonus)

        action = int(np.argmax(scores))
        reward = means[action] + rng.normal(scale=0.25)
        covariance[action] += np.outer(context, context)
        reward_sum[action] += reward * context
        selected.append(action)

    selected = np.asarray(selected)
    regret = np.sum(np.max(expected_rewards, axis=1) - expected_rewards[np.arange(horizon), selected])
    optimal_rate = np.mean(selected == np.argmax(expected_rewards, axis=1))
    return regret, optimal_rate


def run_context_free_ucb():
    counts = np.zeros(actions, dtype=int)
    reward_totals = np.zeros(actions)
    selected = []

    for time_index, means in enumerate(expected_rewards):
        if np.any(counts == 0):
            action = int(np.flatnonzero(counts == 0)[0])
        else:
            scores = reward_totals / counts + np.sqrt(
                2.0 * np.log(time_index + 1) / counts
            )
            action = int(np.argmax(scores))

        reward = means[action] + rng.normal(scale=0.25)
        counts[action] += 1
        reward_totals[action] += reward
        selected.append(action)

    selected = np.asarray(selected)
    regret = np.sum(np.max(expected_rewards, axis=1) - expected_rewards[np.arange(horizon), selected])
    optimal_rate = np.mean(selected == np.argmax(expected_rewards, axis=1))
    return regret, optimal_rate


linucb_regret, linucb_rate = run_linucb()
ucb_regret, ucb_rate = run_context_free_ucb()
print("LinUCB regret / optimal rate:", round(float(linucb_regret), 1), round(float(linucb_rate), 3))
print("context-free UCB:", round(float(ucb_regret), 1), round(float(ucb_rate), 3))
```

</details>

部署上下文 Bandit 时，应记录上下文、可用行动集合、所选行动、奖励定义、反馈延迟、策略版本和**选择倾向概率（selection propensity）**。如果没有行动概率或缺乏充分重叠，就可能无法进行无偏的离策略评估。若上下文本身受到早期行动影响，单步抽象同样会失效，此时可能需要 MDP。


### **马尔可夫决策过程**

**马尔可夫决策过程（Markov decision process, MDP）**用于描述行动会影响未来状态的序列决策。折扣 MDP 通常写作

$$
\mathcal M=(\mathcal S,\mathcal A,P,R,\gamma,\rho_0),
$$

其中，$\mathcal S$ 是状态空间，$\mathcal A$ 是行动空间，$P(s'\mid s,a)$ 是转移核，$R$ 是奖励模型，$\gamma\in[0,1)$ 是折扣因子，$\rho_0$ 是初始状态分布。

马尔可夫假设为

$$
P(S_{t+1},R_{t+1}\mid S_0,A_0,\ldots,S_t,A_t)
=
P(S_{t+1},R_{t+1}\mid S_t,A_t).
$$

它并不是说现实世界没有历史，而是说选定的状态表示已经包含预测下一次转移与奖励所需的全部历史信息。如果仍有未被状态捕获的隐藏信息会影响未来，那么当前观测就不是马尔可夫状态；这时可能需要循环模型、信念状态（belief state）或部分可观测马尔可夫决策过程。

#### **状态、行动、奖励与转移**

- **状态（state）**应当支持预测和控制，而不能只包含方便记录的信息。
- **行动（action）**必须是学习者真正能够干预的决策。
- **奖励（reward）**描述即时偏好，并不是标注“正确行动”的标签。
- **转移（transition）**描述采取行动后未来状态概率如何变化。
- **终止状态（terminal state）**会结束一个回合；由时间上限造成的截断（truncation）在概念上不同，截断处仍可能需要自举估计。

<div class="diagram-scroll">

![一个包含安全与风险转移的小型 MDP。](assets/mdp-transition-graph.svg){fig-alt="起始状态通过安全行动和风险行动，以不同转移概率通向终止目标，说明行动会改变未来状态。"}

</div>

智能体与环境的交互接口明确展示了事件在时间上的先后顺序：

<div class="diagram-scroll">

![强化学习中的智能体与环境交互。](assets/rl-agent-environment.svg){fig-alt="智能体向环境发送行动，环境向智能体返回下一状态与奖励。"}

</div>

*图片来源：[Wikimedia Commons, Agent-environment-diagram-rl.svg](https://commons.wikimedia.org/wiki/File:Agent-environment-diagram-rl.svg)，以 CC0 授权发布。*

不同教材和程序库采用的下标约定可能不同。一种常见写法是

$$
S_t \xrightarrow{A_t} (R_{t+1},S_{t+1}).
$$

应先明确采用哪一种约定，然后始终保持一致。许多看似算法失效的问题，实际上只是一步错位的下标错误。

#### **策略、回报与折扣**

随机策略定义为

$$
\pi(a\mid s)=P(A_t=a\mid S_t=s).
$$

它把状态映射为行动概率分布，与价值函数并不是同一个对象。一条轨迹

$$
\tau=(S_0,A_0,R_1,S_1,A_1,R_2,\ldots)
$$

由初始分布、策略和环境转移共同生成。

从时刻 $t$ 开始的折扣回报为

$$
G_t
=
\sum_{k=0}^{\infty}\gamma^kR_{t+k+1}
=
R_{t+1}+\gamma G_{t+1}.
$$

折扣可以使持续型任务的价值保持有限，表达对即时收益的偏好或对远期结果的不确定性，并控制大约为 $1/(1-\gamma)$ 的有效时间跨度。但不应使用折扣来掩盖错误的回合定义。在有限时域问题中，时间本身可能必须成为状态的一部分，因为随着截止时刻临近，最优行动可能发生变化。

<details>
<summary><strong>Python：在链式 MDP 中模拟轨迹与折扣回报</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(41)
terminal_state = 4
discount = 0.95


def step(state, action):
    # Action 0 intends left; action 1 intends right.
    intended_direction = -1 if action == 0 else 1
    actual_direction = intended_direction if rng.random() < 0.85 else -intended_direction
    next_state = int(np.clip(state + actual_direction, 0, terminal_state))
    reward = 1.0 if next_state == terminal_state else -0.02
    return next_state, reward, next_state == terminal_state


def sample_episode(max_steps=100):
    state = 0
    rewards = []
    states = [state]

    for _ in range(max_steps):
        # A fixed stochastic policy chooses right with probability 0.8.
        action = int(rng.random() < 0.8)
        state, reward, terminated = step(state, action)
        states.append(state)
        rewards.append(reward)
        if terminated:
            break

    discounted_return = sum(
        (discount**time_index) * reward
        for time_index, reward in enumerate(rewards)
    )
    return states, rewards, discounted_return


episodes = [sample_episode() for _ in range(5_000)]
returns = np.array([episode[2] for episode in episodes])
lengths = np.array([len(episode[1]) for episode in episodes])

print("example states:", episodes[0][0])
print("example rewards:", episodes[0][1])
print("mean episode length:", round(float(lengths.mean()), 2))
print("mean discounted return:", round(float(returns.mean()), 3))
print("return standard deviation:", round(float(returns.std(ddof=1)), 3))
```

</details>

奖励设计属于系统规格的一部分。代理奖励可能诱发走捷径、危险探索，或让优化指标看似成功却违背真实目标。应将约束与结果指标分别监控，而不是把所有关注点都压缩成一个标量奖励。


### **Bellman 方程**

Bellman 方程把长期价值分解为“一次即时转移”与“后续价值”。它是递归一致性方程，而不是在 MDP 之外额外加入的一组假设。

#### **状态价值函数与行动价值函数**

对于策略 $\pi$，状态价值函数与行动价值函数分别为

$$
V^\pi(s)
=
\mathbb E_\pi[G_t\mid S_t=s],
$$

$$
Q^\pi(s,a)
=
\mathbb E_\pi[G_t\mid S_t=s,A_t=a].
$$

Bellman 期望方程为

$$
V^\pi(s)
=
\sum_a\pi(a\mid s)
\sum_{s',r}p(s',r\mid s,a)
\left[r+\gamma V^\pi(s')\right],
$$

$$
Q^\pi(s,a)
=
\sum_{s',r}p(s',r\mid s,a)
\left[
r+\gamma\sum_{a'}\pi(a'\mid s')Q^\pi(s',a')
\right].
$$

最优价值函数满足

$$
V^\star(s)
=
\max_a
\sum_{s',r}p(s',r\mid s,a)
\left[r+\gamma V^\star(s')\right],
$$

$$
Q^\star(s,a)
=
\sum_{s',r}p(s',r\mid s,a)
\left[r+\gamma\max_{a'}Q^\star(s',a')\right].
$$

期望方程用于评估一个已指定的策略；最优方程则包含最大化运算，并且是非线性的。混淆两者，就会把“这个策略有多好？”错误地换成“最好的策略是什么？”。

<div class="diagram-scroll">

![Bellman 备份将即时奖励与下一状态价值组合起来。](assets/bellman-backup.svg){fig-alt="当前价值通过一步奖励与折扣后的下一状态价值进行更新；策略评估使用期望，最优控制使用最大值。"}

</div>

对于有限 MDP 和固定策略，

$$
V^\pi=r_\pi+\gamma P_\pi V^\pi,
\qquad
V^\pi=(I-\gamma P_\pi)^{-1}r_\pi.
$$

直接求解适用于小型问题和结果验证；当状态空间很大或转移矩阵稀疏时，迭代方法通常更合适。

<details>
<summary><strong>Python：求解随机策略的 Bellman 期望方程</strong></summary>

```python
import numpy as np

states, actions = 5, 2
terminal = 4
discount = 0.95
transition = np.zeros((states, actions, states))
reward = np.zeros((states, actions, states))

for state in range(states - 1):
    for action in range(actions):
        intended = -1 if action == 0 else 1
        for probability, direction in [(0.85, intended), (0.15, -intended)]:
            next_state = int(np.clip(state + direction, 0, terminal))
            transition[state, action, next_state] += probability
            reward[state, action, next_state] = (
                1.0 if next_state == terminal else -0.02
            )

# Terminal states transition to themselves without additional reward.
transition[terminal, :, terminal] = 1.0
policy = np.tile([0.2, 0.8], (states, 1))

policy_transition = np.einsum("sa,san->sn", policy, transition)
state_action_reward = np.sum(transition * reward, axis=2)
policy_reward = np.sum(policy * state_action_reward, axis=1)

value = np.linalg.solve(
    np.eye(states) - discount * policy_transition,
    policy_reward,
)
action_value = state_action_reward + discount * np.einsum(
    "san,n->sa", transition, value
)

print("V^pi:", np.round(value, 3))
print("Q^pi at start [left, right]:", np.round(action_value[0], 3))
print("policy consistency at start:", round(float(policy[0] @ action_value[0]), 3))
```

</details>

Bellman 残差衡量局部不一致程度：

$$
\delta(s)
=
r_\pi(s)+\gamma(P_\pi V)(s)-V(s).
$$

在被评估模型下，较小的残差是有用的数值检查；但它无法防止转移模型错误、奖励设定错误、隐藏状态或部署分布偏移。


### **动态规划**

动态规划（dynamic programming, DP）假设转移模型与奖励模型已知，并且可以计算期望备份，因此它属于一种**规划（planning）**方法。强化学习则在模型未知时使用采样经验，不过许多无模型算法都可以理解为对动态规划备份的随机近似。

#### **策略评估与策略迭代**

**迭代策略评估（iterative policy evaluation）**反复应用 Bellman 期望算子：

$$
V_{k+1}(s)
\leftarrow
\sum_a\pi(a\mid s)
\sum_{s',r}p(s',r\mid s,a)
\left[r+\gamma V_k(s')\right].
$$

当 $\gamma<1$ 时，该算子在最大范数下是压缩映射，并收敛到唯一的 $V^\pi$。

策略改进定理使用关于 $Q^\pi$ 的贪心行动替换当前策略：

$$
\pi'(s)
\in
\arg\max_a Q^\pi(s,a).
$$

交替执行评估与改进便得到**策略迭代（policy iteration）**。策略评估既可以精确完成，也可以提前停止；修正策略迭代（modified policy iteration）覆盖了完整策略迭代与价值迭代之间的一系列折中方法。

<div class="diagram-scroll">

![策略迭代在评估与改进之间交替进行。](assets/dynamic-programming-loop.svg){fig-alt="策略评估计算当前策略的价值，策略改进使策略变得贪心，并反复循环直到策略稳定。"}

</div>

<details>
<summary><strong>伪代码：策略迭代</strong></summary>

```text
initialize policy pi
repeat:
    evaluate V^pi from Bellman expectation backups
    stable = true
    for each nonterminal state s:
        old_action = pi(s)
        pi(s) = argmax_a sum_{s',r} p(s',r|s,a)[r + gamma V^pi(s')]
        stable = stable and (pi(s) == old_action)
until stable
```

</details>

#### **价值迭代**

价值迭代直接应用 Bellman 最优算子：

$$
V_{k+1}(s)
\leftarrow
\max_a
\sum_{s',r}p(s',r\mid s,a)
\left[r+\gamma V_k(s')\right].
$$

价值收敛后，再从中提取贪心策略。停止容差控制价值估计误差；打印出的策略不再变化，本身并不能证明价值已经收敛。

<details>
<summary><strong>Python：在网格环境中实现策略迭代与价值迭代</strong></summary>

```python
import numpy as np

rows, columns = 4, 4
terminal = rows * columns - 1
discount = 0.95
directions = [(-1, 0), (0, 1), (1, 0), (0, -1)]
symbols = np.array(["U", "R", "D", "L"])


def next_state(state, action):
    if state == terminal:
        return terminal
    row, column = divmod(state, columns)
    row_shift, column_shift = directions[action]
    next_row = int(np.clip(row + row_shift, 0, rows - 1))
    next_column = int(np.clip(column + column_shift, 0, columns - 1))
    return next_row * columns + next_column


def evaluate_policy(policy, tolerance=1e-10):
    value = np.zeros(rows * columns)
    sweeps = 0
    while True:
        updated = value.copy()
        for state in range(rows * columns):
            if state != terminal:
                successor = next_state(state, policy[state])
                updated[state] = -1.0 + discount * value[successor]
        sweeps += 1
        if np.max(np.abs(updated - value)) < tolerance:
            return updated, sweeps
        value = updated


def improve_policy(value):
    policy = np.zeros(rows * columns, dtype=int)
    for state in range(rows * columns):
        if state != terminal:
            action_values = [
                -1.0 + discount * value[next_state(state, action)]
                for action in range(4)
            ]
            policy[state] = int(np.argmax(action_values))
    return policy


def policy_iteration():
    policy = np.zeros(rows * columns, dtype=int)
    improvement_steps = 0
    evaluation_sweeps = 0
    while True:
        value, sweeps = evaluate_policy(policy)
        evaluation_sweeps += sweeps
        improved = improve_policy(value)
        improvement_steps += 1
        if np.array_equal(improved, policy):
            return policy, value, improvement_steps, evaluation_sweeps
        policy = improved


def value_iteration(tolerance=1e-10):
    value = np.zeros(rows * columns)
    sweeps = 0
    while True:
        updated = value.copy()
        for state in range(rows * columns):
            if state != terminal:
                updated[state] = max(
                    -1.0 + discount * value[next_state(state, action)]
                    for action in range(4)
                )
        sweeps += 1
        if np.max(np.abs(updated - value)) < tolerance:
            return improve_policy(updated), updated, sweeps
        value = updated


pi_policy, pi_value, improvement_steps, evaluation_sweeps = policy_iteration()
vi_policy, vi_value, vi_sweeps = value_iteration()

def show_policy(policy):
    labels = symbols[policy].astype(object)
    labels[terminal] = "G"
    return labels.reshape(rows, columns)

print("policy iteration improvements / evaluation sweeps:", improvement_steps, evaluation_sweeps)
print(show_policy(pi_policy))
print("value iteration sweeps:", vi_sweeps)
print(show_policy(vi_policy))
print("maximum value difference:", round(float(np.max(np.abs(pi_value - vi_value))), 10))
```

</details>

同步备份使用前一轮扫描得到的价值；原地更新或优先级更新通常收敛更快，但会改变更新次序。当状态空间很大时，精确枚举将不可行，因此需要采样、函数近似、仿真模型和近似规划。


### **无模型强化学习**

当 $P$ 和 $R$ 未知时，智能体可以根据采样得到的转移估计价值。**无模型（model-free）**表示算法不会学习或使用显式转移模型进行规划；它并不意味着环境没有结构，也不意味着不再需要任何假设。

#### **Monte Carlo 与时序差分学习**

Monte Carlo（MC）策略评估会等待回报 $G_t$ 可用，然后执行更新

$$
V(S_t)
\leftarrow
V(S_t)
+
\alpha\left[G_t-V(S_t)\right].
$$

它不使用自举，但回合回报的方差可能很高，而且必须等到回合终止后才能更新。

一步时序差分学习 TD(0) 在每次转移后更新：

$$
\delta_t
=
R_{t+1}+\gamma V(S_{t+1})-V(S_t),
$$

$$
V(S_t)
\leftarrow
V(S_t)+\alpha\delta_t.
$$

TD 使用采样得到的奖励与下一状态，再从当前的下一状态估计进行自举。当 $V$ 尚不准确时，其目标带有偏差，但通常方差更低，并且可以自然应用于持续型任务。

<div class="diagram-scroll">

![Monte Carlo、TD 与动态规划构造不同的价值目标。](assets/mc-td-backups.svg){fig-alt="Monte Carlo 等待一条采样轨迹的完整回报，TD 使用一次采样转移与自举价值，而动态规划根据已知模型计算期望备份。"}

</div>

这里的核心区别是**采样还是求期望**，以及**自举还是使用完整回报**。多步 TD 方法和资格迹（eligibility trace）在 Monte Carlo 与一步 TD 之间形成连续过渡。

<details>
<summary><strong>Python：在随机游走问题上比较 Monte Carlo 与 TD(0)</strong></summary>

```python
import numpy as np

nonterminal_states = np.arange(1, 6)
true_value = nonterminal_states / 6.0
episodes = 200
runs = 100
checkpoints = [1, 10, 50, 200]


def generate_episode(rng):
    state = 3
    trajectory = []
    while state not in (0, 6):
        next_state = state + (-1 if rng.random() < 0.5 else 1)
        reward = 1.0 if next_state == 6 else 0.0
        trajectory.append((state, reward, next_state))
        state = next_state
    return trajectory


def learn(method, seed, alpha=0.10):
    rng = np.random.default_rng(seed)
    value = np.full(7, 0.5)
    value[[0, 6]] = [0.0, 0.0]
    errors = []

    for episode_index in range(1, episodes + 1):
        trajectory = generate_episode(rng)

        if method == "mc":
            returns = np.zeros(len(trajectory))
            running_return = 0.0
            for index in range(len(trajectory) - 1, -1, -1):
                running_return = trajectory[index][1] + running_return
                returns[index] = running_return

            visited = set()
            for index, (state, _, _) in enumerate(trajectory):
                if state not in visited:  # first-visit MC
                    value[state] += alpha * (returns[index] - value[state])
                    visited.add(state)

        elif method == "td":
            for state, reward, next_state in trajectory:
                bootstrap = 0.0 if next_state in (0, 6) else value[next_state]
                target = reward + bootstrap
                value[state] += alpha * (target - value[state])
        else:
            raise ValueError("unknown method")

        errors.append(np.sqrt(np.mean((value[1:6] - true_value) ** 2)))

    return np.asarray(errors)


for method in ["mc", "td"]:
    error_curves = np.array(
        [learn(method, seed=50_000 + run) for run in range(runs)]
    )
    summary = {
        checkpoint: round(float(error_curves[:, checkpoint - 1].mean()), 3)
        for checkpoint in checkpoints
    }
    print(method.upper(), "mean RMSE by episode:", summary)
```

</details>

没有一种方法在所有问题上都占优。当回合较短且完整回报可信时，Monte Carlo 很有吸引力；对于很长或持续不断的轨迹以及在线更新，TD 更适合。比较时应使用多个独立随机种子的学习曲线，并评估对 $\alpha$、初始化、回合截断与奖励尺度的敏感性。

#### **SARSA 与 Q-learning**

为了实现控制，算法必须在策略不断变化的同时学习状态—行动对的价值。

**SARSA** 是同策略（on-policy）算法：

$$
Q(S_t,A_t)
\leftarrow
Q(S_t,A_t)
+
\alpha
\left[
R_{t+1}
+\gamma Q(S_{t+1},A_{t+1})
-Q(S_t,A_t)
\right].
$$

它的目标使用行为策略实际选出的下一个行动。当持续采用 $\epsilon$-greedy 探索时，它会把探索行动造成的后果也计入价值。

**Q-learning** 是离策略（off-policy）算法：

$$
Q(S_t,A_t)
\leftarrow
Q(S_t,A_t)
+
\alpha
\left[
R_{t+1}
+\gamma\max_aQ(S_{t+1},a)
-Q(S_t,A_t)
\right].
$$

它的行为策略可以探索，但更新目标按照贪心策略行动。在表格型设置中，只要探索充分且步长采用适当的递减方式，Q-learning 可以收敛到 $Q^\star$。

<div class="diagram-scroll">

![SARSA 与 Q-learning 在悬崖行走问题中可能选择不同路线。](assets/sarsa-qlearning-cliff.svg){fig-alt="SARSA 的同策略目标考虑探索时跌落悬崖的风险，因此学习更安全的路线；Q-learning 的贪心目标则偏好沿悬崖前进的最短路线。"}

</div>

<details>
<summary><strong>Python：在悬崖行走环境中比较 SARSA 与 Q-learning</strong></summary>

```python
import numpy as np

rows, columns, actions = 4, 12, 4
start = (3, 0)
goal = (3, 11)
cliff = {(3, column) for column in range(1, 11)}
directions = [(-1, 0), (0, 1), (1, 0), (0, -1)]


def environment_step(state, action):
    row, column = state
    row_shift, column_shift = directions[action]
    candidate = (
        int(np.clip(row + row_shift, 0, rows - 1)),
        int(np.clip(column + column_shift, 0, columns - 1)),
    )
    if candidate in cliff:
        return start, -100.0, False, True
    if candidate == goal:
        return goal, -1.0, True, False
    return candidate, -1.0, False, False


def epsilon_greedy(q_values, state, rng, epsilon=0.10):
    if rng.random() < epsilon:
        return int(rng.integers(actions))
    row, column = state
    best = np.flatnonzero(q_values[row, column] == q_values[row, column].max())
    return int(rng.choice(best))


def train(method, seed, episodes=300, alpha=0.50, discount=1.0):
    rng = np.random.default_rng(seed)
    q_values = np.zeros((rows, columns, actions))
    episode_returns, cliff_falls = [], []

    for _ in range(episodes):
        state = start
        action = epsilon_greedy(q_values, state, rng)
        total_reward = 0.0
        falls = 0

        for _ in range(1_000):
            next_state, reward, terminated, fell = environment_step(state, action)
            total_reward += reward
            falls += int(fell)

            if terminated:
                target = reward
                next_action = None
            elif method == "sarsa":
                next_action = epsilon_greedy(q_values, next_state, rng)
                target = reward + discount * q_values[next_state][next_action]
            elif method == "q-learning":
                next_action = epsilon_greedy(q_values, next_state, rng)
                target = reward + discount * np.max(q_values[next_state])
            else:
                raise ValueError("unknown method")

            q_values[state][action] += alpha * (
                target - q_values[state][action]
            )
            if terminated:
                break
            state, action = next_state, next_action

        episode_returns.append(total_reward)
        cliff_falls.append(falls)

    return q_values, np.asarray(episode_returns), np.asarray(cliff_falls)


for method in ["sarsa", "q-learning"]:
    outcomes = [
        train(method, seed=70_000 + run)[1:]
        for run in range(8)
    ]
    returns = np.array([outcome[0] for outcome in outcomes])
    falls = np.array([outcome[1] for outcome in outcomes])
    print(
        method,
        "last-30 mean return =", round(float(returns[:, -30:].mean()), 1),
        "cliff falls / episode =", round(float(falls[:, -30:].mean()), 3),
    )
```

</details>

“同策略”与“离策略”描述的是生成数据的策略和被评估或改进的策略之间的关系。它们并不表示在线与离线，也不自动保证安全性。离策略方法需要目标行动得到充分覆盖；当函数近似、自举和离策略更新同时出现时，训练可能变得不稳定。


### **函数近似与深度强化学习**

表格型方法为每个状态或状态—行动对存储一个价值。当状态是图像、连续测量值、长历史或组合配置时，这种方式将不可行。函数近似用下列函数替代表格：

$$
\hat V(s;w),
\qquad
\hat Q(s,a;w),
$$

因此在一个状态中获得的经验可以泛化到相似状态。

这种泛化同时引入了近似误差与优化不稳定性。下列三者的组合

1. **函数近似（function approximation）**；
2. **自举（bootstrapping）**；
3. **离策略学习（off-policy learning）**

被称为**致命三角（deadly triad）**，因为即使每一项单独使用都很有价值，它们组合后仍可能导致价值估计发散。神经网络还会进一步引入非凸优化、轨迹样本的相关性、持续变化的数据分布，以及对覆盖不足行动的外推。

#### **DQN：连接强化学习与深度学习**

深度 Q 网络（Deep Q-Network, DQN）最小化时序差分损失

$$
\mathcal L(\theta)
=
\mathbb E_{(s,a,r,s',d)\sim\mathcal B}
\left[
\operatorname{Huber}
\left(
y-Q_\theta(s,a)
\right)
\right],
$$

其目标为

$$
y
=
r
+
\gamma(1-d)\max_{a'}Q_{\theta^-}(s',a').
$$

这里，$d$ 标记真正的终止转移，$\mathcal B$ 是经验回放分布，$\theta^-$ 是延迟更新的目标网络参数副本。

<div class="diagram-scroll">

![DQN 组合在线网络、经验回放缓冲区、目标网络与 TD 损失。](assets/dqn-stabilizers.svg){fig-alt="转移样本进入经验回放缓冲区，随机小批量用于训练在线 Q 网络，延迟更新的目标网络提供更稳定的自举目标。"}

</div>

主要稳定机制各自解决不同问题：

- **经验回放（experience replay）**复用数据，并减弱短时间范围内的样本相关性；
- **目标网络（target network）**减缓自举目标的移动速度；
- **Huber 损失与梯度裁剪**降低算法对巨大 TD 误差的敏感度；
- **奖励缩放或裁剪**控制数值范围，但处理不当会改变原始目标；
- **Double DQN**将行动选择与目标评估分离，以减轻最大化偏差。

经验回放使 DQN 成为离策略算法，但回放数据仍然必须覆盖足够多的行动。神经网络可能为数据中从未出现的行动分配任意高的价值，这正是离线强化学习的核心难题之一。

<details>
<summary><strong>Python：使用经验回放与目标网络训练一个紧凑的 DQN</strong></summary>

```python
from collections import deque
import random
import numpy as np
import torch
from torch import nn

torch.manual_seed(17)
np.random.seed(17)
random.seed(17)
torch.set_num_threads(1)

state_count, action_count = 7, 2
terminal_state = state_count - 1


def transition(state, action, rng):
    # A short chain with 10% action slip and a goal on the right.
    intended = -1 if action == 0 else 1
    direction = intended if rng.random() > 0.10 else -intended
    next_state = int(np.clip(state + direction, 0, terminal_state))
    terminated = next_state == terminal_state
    reward = 1.0 if terminated else -0.02
    return next_state, reward, terminated


def one_hot(state):
    vector = np.zeros(state_count, dtype=np.float32)
    vector[state] = 1.0
    return vector


def make_network():
    return nn.Sequential(
        nn.Linear(state_count, 32),
        nn.ReLU(),
        nn.Linear(32, action_count),
    )


online_network = make_network()
target_network = make_network()
target_network.load_state_dict(online_network.state_dict())
optimizer = torch.optim.Adam(online_network.parameters(), lr=2e-3)
replay = deque(maxlen=5_000)
rng = np.random.default_rng(17)
optimization_steps = 0

for episode in range(250):
    state = 0
    epsilon = max(0.05, 1.0 - episode / 200)

    for _ in range(40):
        if rng.random() < epsilon:
            action = int(rng.integers(action_count))
        else:
            with torch.no_grad():
                action = int(
                    online_network(torch.tensor(one_hot(state))).argmax()
                )

        next_state, reward, terminated = transition(state, action, rng)
        replay.append((state, action, reward, next_state, terminated))
        state = next_state

        if len(replay) >= 64:
            batch = random.sample(replay, 32)
            states, actions, rewards, next_states, terminals = zip(*batch)
            state_tensor = torch.tensor(
                np.stack([one_hot(item) for item in states])
            )
            next_state_tensor = torch.tensor(
                np.stack([one_hot(item) for item in next_states])
            )
            action_tensor = torch.tensor(actions, dtype=torch.long)
            reward_tensor = torch.tensor(rewards, dtype=torch.float32)
            terminal_tensor = torch.tensor(terminals, dtype=torch.float32)

            predicted_q = online_network(state_tensor).gather(
                1, action_tensor[:, None]
            ).squeeze(1)
            with torch.no_grad():
                next_q = target_network(next_state_tensor).max(dim=1).values
                target_q = reward_tensor + 0.97 * (1.0 - terminal_tensor) * next_q

            loss = nn.functional.smooth_l1_loss(predicted_q, target_q)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(online_network.parameters(), 5.0)
            optimizer.step()
            optimization_steps += 1

            if optimization_steps % 75 == 0:
                target_network.load_state_dict(online_network.state_dict())

        if terminated:
            break


def evaluate(episodes=300):
    evaluation_rng = np.random.default_rng(999)
    returns, successes = [], []
    for _ in range(episodes):
        state, total_reward = 0, 0.0
        for _ in range(40):
            with torch.no_grad():
                action = int(
                    online_network(torch.tensor(one_hot(state))).argmax()
                )
            state, reward, terminated = transition(
                state, action, evaluation_rng
            )
            total_reward += reward
            if terminated:
                break
        returns.append(total_reward)
        successes.append(state == terminal_state)
    return np.asarray(returns), np.asarray(successes)


evaluation_returns, successes = evaluate()
with torch.no_grad():
    start_q = online_network(torch.tensor(one_hot(0))).numpy()
print("greedy success rate:", round(float(successes.mean()), 3))
print("mean undiscounted return:", round(float(evaluation_returns.mean()), 3))
print("Q(start, left/right):", np.round(start_q, 3))
print("replay transitions:", len(replay))
```

</details>

本例刻意保持较小规模，以便逐步审查。大型深度强化学习实验还需要多个随机种子、完整学习曲线、环境与包装器版本、关闭探索噪声后的评估、运行时间与样本预算，以及对经验回放、目标更新、奖励处理和网络结构的消融实验。单个随机种子得到的高最终分数并不是可靠证据。


### **策略优化**

基于价值的方法通过最大化估计行动价值来间接得到策略。**策略优化（policy optimization）**则直接参数化策略，尤其适用于连续行动、随机行为、受约束的分布，或必须显式建模概率密度的策略。

#### **策略梯度与 Actor-Critic 方法**

设可微策略 $\pi_\theta$ 生成轨迹 $\tau$，目标为

$$
J(\theta)
=
\mathbb E_{\tau\sim\pi_\theta}
\left[
\sum_{t=0}^{T-1}\gamma^tR_{t+1}
\right].
$$

策略梯度定理给出如下形式：

$$
\nabla_\theta J(\theta)
\propto
\mathbb E_{\pi_\theta}
\left[
\nabla_\theta\log\pi_\theta(A_t\mid S_t)
Q^{\pi_\theta}(S_t,A_t)
\right].
$$

对数概率项会依据行动的估计长期质量，提高或降低该行动的概率。**REINFORCE** 使用采样回报替代 $Q^\pi$。在其采样假设成立时，这一估计量无偏，但方差通常很高。

减去基线 $b(s)$ 不会改变梯度的期望：

$$
\mathbb E
\left[
\nabla_\theta\log\pi_\theta(A_t\mid S_t)
\left(G_t-b(S_t)\right)
\right].
$$

令 $b(s)\approx V^\pi(s)$，便可得到优势估计。一个 **Actor-Critic** 方法会同时学习：

- 负责选择行动的 **Actor** $\pi_\theta(a\mid s)$；
- 负责估计价值的 **Critic** $V_\phi(s)$ 或 $Q_\phi(s,a)$；
- 用于训练 Actor 的优势或 TD 误差信号。

对于一步 Critic，

$$
\delta_t
=
R_{t+1}
+\gamma V_\phi(S_{t+1})
-V_\phi(S_t)
$$

可以作为优势估计。

<div class="diagram-scroll">

![Actor 选择行动，Critic 估计行动优势。](assets/actor-critic-loop.svg){fig-alt="Actor 向环境发送行动，Critic 评估得到的转移，并用优势信号更新 Actor。"}

</div>

<details>
<summary><strong>Python：在链式环境中训练一步 Actor-Critic</strong></summary>

```python
import numpy as np
import torch
from torch import nn
from torch.distributions import Categorical

torch.manual_seed(29)
torch.set_num_threads(1)
rng = np.random.default_rng(29)

state_count, action_count = 7, 2
terminal_state = state_count - 1


def encode(state):
    observation = torch.zeros(state_count)
    observation[state] = 1.0
    return observation


def step(state, action):
    intended = -1 if action == 0 else 1
    direction = intended if rng.random() > 0.10 else -intended
    next_state = int(np.clip(state + direction, 0, terminal_state))
    terminated = next_state == terminal_state
    reward = 1.0 if terminated else -0.02
    return next_state, reward, terminated


actor = nn.Sequential(
    nn.Linear(state_count, 24),
    nn.Tanh(),
    nn.Linear(24, action_count),
)
critic = nn.Sequential(
    nn.Linear(state_count, 24),
    nn.Tanh(),
    nn.Linear(24, 1),
)
optimizer = torch.optim.Adam(
    list(actor.parameters()) + list(critic.parameters()),
    lr=3e-3,
)
discount = 0.97

for episode in range(250):
    state = 0
    for _ in range(40):
        observation = encode(state)
        distribution = Categorical(logits=actor(observation))
        action = distribution.sample()
        next_state, reward, terminated = step(state, int(action))

        value = critic(observation).squeeze()
        with torch.no_grad():
            next_value = (
                torch.tensor(0.0)
                if terminated
                else critic(encode(next_state)).squeeze()
            )
            td_target = torch.tensor(reward) + discount * next_value

        advantage = td_target - value
        actor_loss = (
            -distribution.log_prob(action) * advantage.detach()
            - 0.01 * distribution.entropy()
        )
        critic_loss = 0.5 * advantage.pow(2)

        optimizer.zero_grad()
        (actor_loss + critic_loss).backward()
        nn.utils.clip_grad_norm_(
            list(actor.parameters()) + list(critic.parameters()), 5.0
        )
        optimizer.step()

        state = next_state
        if terminated:
            break


def evaluate_policy(episodes=300):
    evaluation_rng = np.random.default_rng(1_029)
    successes, returns = [], []
    for _ in range(episodes):
        state, total_reward = 0, 0.0
        for _ in range(40):
            with torch.no_grad():
                action = int(actor(encode(state)).argmax())
            intended = -1 if action == 0 else 1
            direction = intended if evaluation_rng.random() > 0.10 else -intended
            state = int(np.clip(state + direction, 0, terminal_state))
            terminated = state == terminal_state
            reward = 1.0 if terminated else -0.02
            total_reward += reward
            if terminated:
                break
        successes.append(state == terminal_state)
        returns.append(total_reward)
    return np.mean(successes), np.mean(returns)


success_rate, mean_return = evaluate_policy()
with torch.no_grad():
    start_probabilities = torch.softmax(actor(encode(0)), dim=0).numpy()
    start_value = float(critic(encode(0)))
print("greedy success rate:", round(float(success_rate), 3))
print("mean undiscounted return:", round(float(mean_return), 3))
print("policy at start [left, right]:", np.round(start_probabilities, 3))
print("critic value at start:", round(start_value, 3))
```

</details>

Actor-Critic 方法引入了相互耦合的近似：Actor 会改变 Critic 所看到的数据分布，而 Critic 的误差又会改变 Actor 的更新。熵奖励、广义优势估计、信赖域、裁剪、回放修正和目标 Critic 分别用于处理不同的失败模式。算法名称不能代替诊断；应检查回报、回合长度、熵、KL 散度、价值损失、优势尺度、约束违反情况以及不同随机种子之间的变化。


### **探索、离策略评估与安全**

MDP 中的探索比 Bandit 更困难，因为获取信息可能需要连续采取一系列行动，奖励可能延迟出现，而且一次不安全的行动就可能让智能体进入不可逆状态。常见机制包括：

- 随机行动噪声或熵正则化；
- 乐观估计与置信奖励；
- 基于计数或伪计数的新颖度；
- 基于预测误差或信息增益的内在奖励；
- 对模型或价值函数进行后验采样；
- 示范数据、课程学习、模拟器或重置机制。

更高的新颖度并不一定更好。预测误差可能奖励随机噪声；行动噪声在受约束空间中可能没有意义；内在目标也可能使策略偏离部署时的真实目标。应分别评估状态覆盖率、任务回报、约束违反情况和数据收集成本。

当候选策略 $\pi$ 必须根据另一个行为策略 $b$ 生成的数据进行评估时，这个问题称为**离策略评估（off-policy evaluation, OPE）**。在上下文 Bandit 中，设日志元组为 $(x_i,a_i,r_i,p_i)$，且已知倾向概率 $p_i=b(a_i\mid x_i)$，逆倾向评分估计为

$$
\hat V_{\mathrm{IPS}}(\pi)
=
\frac{1}{n}
\sum_{i=1}^{n}
\frac{\pi(a_i\mid x_i)}{b(a_i\mid x_i)}
r_i.
$$

在倾向概率正确、一致性成立且具有重叠性的条件下，IPS 是无偏的，但方差可能极大。自归一化估计量再除以权重之和，以有限样本偏差换取稳定性。

直接奖励模型 $\hat q(x,a)$ 通过下式估计策略价值：

$$
\hat V_{\mathrm{DM}}(\pi)
=
\frac{1}{n}\sum_i\sum_a\pi(a\mid x_i)\hat q(x_i,a).
$$

双重稳健估计量将两者结合：

$$
\hat V_{\mathrm{DR}}(\pi)
=
\frac{1}{n}\sum_i
\left[
\sum_a\pi(a\mid x_i)\hat q(x_i,a)
+
\frac{\pi(a_i\mid x_i)}{b(a_i\mid x_i)}
\left(r_i-\hat q(x_i,a_i)\right)
\right].
$$

当同一份日志既用于训练 $\hat q$，又用于估计策略价值时，交叉拟合（cross-fitting）可以减轻过拟合偏差。

<details>
<summary><strong>Python：比较直接法、IPS、自归一化与双重稳健 OPE</strong></summary>

```python
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold

rng = np.random.default_rng(53)
sample_count, dimensions, actions = 20_000, 5, 3
contexts = rng.normal(size=(sample_count, dimensions))
parameters = np.array(
    [
        [0.9, -0.5, 0.3, 0.0, 0.2],
        [-0.4, 0.8, 0.1, 0.5, -0.2],
        [0.2, -0.1, 0.9, -0.6, 0.4],
    ]
)


def sigmoid(values):
    return 1.0 / (1.0 + np.exp(-values))


def softmax(values):
    shifted = values - values.max(axis=1, keepdims=True)
    exponentiated = np.exp(shifted)
    return exponentiated / exponentiated.sum(axis=1, keepdims=True)


true_reward_probability = sigmoid(contexts @ parameters.T)

# Both policies retain positive probability for every action.
behavior_base = softmax(1.2 * true_reward_probability + np.array([0.3, 0.0, -0.2]))
behavior_policy = 0.15 / actions + 0.85 * behavior_base
target_base = softmax(5.0 * true_reward_probability)
target_policy = 0.05 / actions + 0.95 * target_base

uniforms = rng.random(sample_count)
logged_actions = (
    uniforms[:, None] > np.cumsum(behavior_policy, axis=1)
).sum(axis=1)
logged_probabilities = behavior_policy[np.arange(sample_count), logged_actions]
logged_rewards = rng.binomial(
    1,
    true_reward_probability[np.arange(sample_count), logged_actions],
)

# Cross-fitted outcome predictions: each row is predicted by a model that
# did not train on that row.
predicted_rewards = np.zeros((sample_count, actions))
folds = KFold(n_splits=2, shuffle=True, random_state=53)
for train_index, validation_index in folds.split(contexts):
    for action in range(actions):
        action_train = train_index[logged_actions[train_index] == action]
        model = LogisticRegression(max_iter=1_000)
        model.fit(contexts[action_train], logged_rewards[action_train])
        predicted_rewards[validation_index, action] = model.predict_proba(
            contexts[validation_index]
        )[:, 1]

target_probability_logged = target_policy[
    np.arange(sample_count), logged_actions
]
weights = target_probability_logged / logged_probabilities
direct_per_row = np.sum(target_policy * predicted_rewards, axis=1)

direct = np.mean(direct_per_row)
ips = np.mean(weights * logged_rewards)
self_normalized = np.sum(weights * logged_rewards) / np.sum(weights)
doubly_robust = np.mean(
    direct_per_row
    + weights
    * (
        logged_rewards
        - predicted_rewards[np.arange(sample_count), logged_actions]
    )
)
true_value = np.mean(
    np.sum(target_policy * true_reward_probability, axis=1)
)
effective_sample_size = np.sum(weights) ** 2 / np.sum(weights**2)

print("true target-policy value:", round(float(true_value), 4))
print("direct / IPS / SNIPS / DR:", np.round(
    [direct, ips, self_normalized, doubly_robust], 4
))
print("99th percentile / max weight:", np.round(
    [np.quantile(weights, 0.99), weights.max()], 2
))
print("effective sample size:", round(float(effective_sample_size), 1))
```

</details>

在多步强化学习中，轨迹重要性比率会沿时间相乘，方差可能随时域长度呈指数增长。逐决策加权、拟合价值方法、基于模型的估计量和序列型双重稳健估计量各自引入不同的偏差—方差权衡。任何估计量都无法恢复行为策略从未访问过的行动或状态所对应的信息。

<div class="diagram-scroll">

![离策略评估与安全检查共同构成部署门槛。](assets/ope-safety-gates.svg){fig-alt="日志数据依次通过重叠性与不确定性诊断、显式安全约束和分阶段部署门槛，之后策略才会被广泛发布。"}

</div>

安全性应被显式建模。约束 MDP 可以求解

$$
\max_\pi J_R(\pi)
\quad\text{subject to}\quad
J_{C_j}(\pi)\le d_j,\qquad j=1,\ldots,m,
$$

其中，$C_j$ 分别衡量碰撞、不公平曝光、延迟、能耗或人工干预等成本。除非能够论证所采用的权衡，否则奖励惩罚并不等同于硬约束。

一个可辩护的策略开发流程包括：

1. 不可随意更改的日志记录与数据质量规范；
2. 重叠性与有效样本量诊断；
3. 验证模拟器，但不把模拟器当作现实真值；
4. 使用不确定性区间和敏感性分析进行离线策略评估；
5. 设置显式约束、人工接管、速率限制与回滚机制；
6. 在广泛部署前进行小规模、受监控的上线；
7. 部署后监测漂移、奖励、成本和各子群体表现。

奖励投机、分布偏移、未观测混杂、延迟效应和反馈循环既是算法问题，也是规格定义问题。人工监督必须拥有真正暂停或撤销策略的权限。


### **选择序列学习建模形式**

应从能够描述行动真实后果的最弱建模形式开始：

<div class="diagram-scroll">

![选择在线学习、Bandit 或强化学习的决策树。](assets/sequential-formulation-tree.svg){fig-alt="如果行动会影响未来状态，则使用 MDP；否则，只能观察所选行动反馈时使用 Bandit，能够观察完整反馈时使用在线学习。"}

</div>

| 问题 | 如果是 | 如果否 |
|---|---|---|
| 是否必须在全部目标值可用前作出决策？ | 考虑序列学习协议 | 批量学习可能已经足够 |
| 行动是否会改变后续状态或机会？ | MDP / 强化学习 | 继续检查反馈可见性 |
| 是否只能观察所选行动的奖励？ | Bandit 或上下文 Bandit | 全信息在线学习 |
| 是否在行动前观察到上下文？ | 上下文策略 | 无上下文 Bandit |
| 是否有可信的转移模型？ | 规划或基于模型的强化学习 | 无模型方法或学习模型的方法 |
| 能否在线探索不安全行动？ | 仍需施加约束并持续监控 | 优先使用日志数据、模拟、示范或保守部署 |

选择算法前，应明确写下：

1. 状态、上下文、行动、奖励与终止信号的基本单位及出现时刻；
2. 行动是否会因果性地影响未来观测；
3. 未被选择的行动有哪些反馈可见；
4. 比较对象、回报、时域、折扣与约束；
5. 生成训练数据的行为策略及其倾向概率；
6. 有效的泛化单位：未来时间、新用户、新环境还是新任务；
7. 基线策略、部署门槛、监控计划与回滚路径。

随后可以比较主要算法家族：

| 方法 | 是否需要模型？ | 同策略/离策略 | 是否自举？ | 主要优势 | 主要风险 |
|---|---:|---|---:|---|---|
| OGD / 在线 Perceptron | 否 | 全信息数据流 | 否 | 在较弱序列假设下仍有遗憾界 | 比较对象错误或无法及时跟踪漂移 |
| UCB / Thompson sampling | 否 | 在线 Bandit 交互 | 否 | 高效的一步探索 | 平稳性与安全假设 |
| 动态规划 | 是 | 规划，而非采样行为 | 是 | 精确的表格型解 | 依赖模型和状态空间规模 |
| Monte Carlo | 否 | 通常用于同策略评估 | 否 | 使用完整回报作为目标 | 方差高且更新延迟 |
| SARSA | 否 | 同策略 | 是 | 将探索行为的后果计入价值 | 样本效率与策略依赖 |
| Q-learning / DQN | 否 | 离策略 | 是 | 可复用数据并以贪心控制为目标 | 外推误差与训练不稳定 |
| 策略梯度 | 否 | 通常为同策略 | 取决于使用回报还是 Critic | 直接优化随机策略或连续策略 | 方差高且样本成本大 |
| Actor-Critic | 否 | 可为两者，离策略时需要修正 | 是 | 以较低方差学习策略 | Actor 与 Critic 的误差相互耦合 |

只有当序列方法在**真实反馈与后果结构**下改进决策时，才能称为成功；模拟器中的高回报并不足够。应使用简单基线、多个随机种子、学习曲线、不确定性区间、资源消耗统计、约束指标和分阶段部署。

主要及官方资源包括 [A Modern Introduction to Online Learning](https://arxiv.org/abs/1912.13213)、免费在线教材 [Bandit Algorithms](https://tor-lattimore.com/downloads/book/book.pdf)、最初的 [UCB1 分析](https://doi.org/10.1023/A:1013689704352)、Sutton 与 Barto 的 [Reinforcement Learning: An Introduction](https://mitpress.mit.edu/9780262039246/reinforcement-learning/)、[Stanford CS234](https://web.stanford.edu/class/cs234/modules.html)、[Berkeley CS285](https://rail.eecs.berkeley.edu/deeprlcourse-fa23/)、[OpenAI Spinning Up](https://spinningup.openai.com/en/latest/)、最初的 [DQN 论文](https://doi.org/10.1038/nature14236)、[策略梯度定理论文](https://proceedings.neurips.cc/paper/1999/hash/464d828b85b0bed98e80ade0a5c43b0f-Abstract.html)、[双重稳健策略评估](https://arxiv.org/abs/1103.4601)以及 [Constrained Policy Optimization](https://proceedings.mlr.press/v70/achiam17a.html)。
